#  Segmentasi dan Prediksi Risiko Diabetes
## Menggunakan K-Means Clustering dan Random Forest Classifier
---
**Dataset:** Pima Indians Diabetes Database  
**Metode:** K-Means (Clustering) + Random Forest (Classification)  
**Tujuan:** Mengelompokkan pasien berdasarkan profil kesehatan dan memprediksi risiko diabetes

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from sklearn.decomposition import PCA
import joblib
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../dataset/diabetes.csv')
print("Shape:", df.shape)
df.head(10)

## 3. Exploratory Data Analysis (EDA)

### 3.1 Statistik Deskriptif

In [ ]:
df.describe().round(2)

### 3.2 Cek Missing Values dan Tipe Data

In [ ]:
print("Informasi Dataset:")
print(df.info())
print()
print("Null values per kolom:")
print(df.isnull().sum())
print()
print("Nilai nol pada kolom medis (tidak valid secara klinis):")
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_cols:
    zeros = (df[col] == 0).sum()
    pct = zeros/len(df)*100
    print(f"  {col:28s}: {zeros} ({pct:.1f}%)")

### 3.3 Distribusi Kelas Outcome

In [ ]:
plt.figure(figsize=(6, 5))
counts = df['Outcome'].value_counts()
bars = plt.bar(['Tidak Diabetes (0)', 'Diabetes (1)'],
               counts.values, color=['#2ecc71', '#e74c3c'],
               edgecolor='black', alpha=0.8, width=0.5)
for bar, count in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(count), ha='center', fontsize=13, fontweight='bold')
plt.title('Distribusi Kelas Outcome', fontsize=14, fontweight='bold')
plt.ylabel('Jumlah Data')
plt.ylim(0, 600)
plt.tight_layout()
plt.show()

print(f"Tidak Diabetes: {counts[0]} ({counts[0]/len(df)*100:.1f}%)")
print(f"Diabetes      : {counts[1]} ({counts[1]/len(df)*100:.1f}%)")

### 3.4 Histogram Distribusi Fitur

In [ ]:
features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
            'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()
for i, col in enumerate(features):
    axes[i].hist(df[col], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Distribusi {col}', fontsize=10)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frekuensi')
axes[8].axis('off')
plt.suptitle('Distribusi Fitur Dataset Pima Indians Diabetes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.5 Heatmap Korelasi

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Heatmap Korelasi Fitur', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Fitur paling berkorelasi dengan Outcome:")
print(corr['Outcome'].sort_values(ascending=False).drop('Outcome'))

## 4. Data Preprocessing

In [ ]:
# Imputasi nilai nol yang tidak valid secara medis
df_clean = df.copy()
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_cols:
    median_val = df_clean[df_clean[col] != 0][col].median()
    count_zeros = (df_clean[col] == 0).sum()
    df_clean[col] = df_clean[col].replace(0, median_val)
    print(f"{col:28s}: {count_zeros} nilai nol → diisi median ({median_val:.1f})")

print()
print("Verifikasi NaN setelah imputasi:", df_clean.isnull().sum().sum())

In [ ]:
# Standardisasi fitur
X = df_clean.drop('Outcome', axis=1)
y = df_clean['Outcome']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Total data     : {len(df_clean)}")
print(f"Training set   : {len(X_train)} ({len(X_train)/len(df_clean)*100:.0f}%)")
print(f"Testing set    : {len(X_test)} ({len(X_test)/len(df_clean)*100:.0f}%)")

## 5. Clustering dengan K-Means

### 5.1 Menentukan K Optimal (Elbow Method)

In [ ]:
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=3, color='red', linestyle='--', linewidth=2, label='K optimal = 3')
plt.fill_between(list(K_range), inertias, alpha=0.1, color='blue')
plt.xlabel('Jumlah Cluster (K)', fontsize=12)
plt.ylabel('Inertia (Within-cluster Sum of Squares)', fontsize=12)
plt.title('Elbow Method untuk Menentukan K Optimal', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("K optimal berdasarkan Elbow Method: K = 3")

### 5.2 Training K-Means (K=3)

In [ ]:
# Training K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)
df_clean['Cluster'] = cluster_labels

# Tentukan label risiko berdasarkan rata-rata outcome
outcome_means = df_clean.groupby('Cluster')['Outcome'].mean()
sorted_clusters = outcome_means.sort_values().index.tolist()
risk_labels = ['Risiko Rendah', 'Risiko Sedang', 'Risiko Tinggi']
risk_map = {c: risk_labels[i] for i, c in enumerate(sorted_clusters)}

print("=== Cluster Risk Mapping ===")
for c, label in risk_map.items():
    count = (cluster_labels == c).sum()
    print(f"Cluster {c}: {label:15s} | Jumlah: {count:3d} | Diabetes rate: {outcome_means[c]:.2%}")

### 5.3 Analisis Karakteristik Cluster

In [ ]:
cluster_analysis = df_clean.groupby('Cluster').agg({
    'Pregnancies': 'mean', 'Glucose': 'mean', 'BloodPressure': 'mean',
    'Insulin': 'mean', 'BMI': 'mean', 'Age': 'mean', 'Outcome': ['mean', 'count']
}).round(2)
cluster_analysis.columns = ['Pregnancies', 'Glucose', 'BloodPressure',
                              'Insulin', 'BMI', 'Age', 'Diabetes Rate', 'Count']
cluster_analysis['Label Risiko'] = [risk_map[i] for i in cluster_analysis.index]
print(cluster_analysis.to_string())

### 5.4 Visualisasi Cluster (PCA 2D)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

colors = ['#2ecc71', '#f39c12', '#e74c3c']
plt.figure(figsize=(10, 7))
for i in range(3):
    mask = cluster_labels == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=colors[i], label=f'Cluster {i}: {risk_map[i]}',
                alpha=0.6, s=60, edgecolors='black', linewidth=0.3)

plt.title('Visualisasi Cluster K-Means (PCA 2D)', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance explained)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance explained)')
plt.legend(fontsize=11, loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Classification dengan Random Forest

### 6.1 Training Model

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42,
                           max_depth=10, min_samples_split=5)
rf.fit(X_train, y_train)
print("Random Forest berhasil ditraining!")
print(f"Jumlah pohon : {rf.n_estimators}")
print(f"Max depth    : {rf.max_depth}")

### 6.2 Evaluasi Model

In [ ]:
y_pred = rf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("=" * 40)
print("  HASIL EVALUASI RANDOM FOREST")
print("=" * 40)
print(f"  Accuracy  : {acc*100:.2f}%")
print(f"  Precision : {prec*100:.2f}%")
print(f"  Recall    : {rec*100:.2f}%")
print(f"  F1-Score  : {f1*100:.2f}%")
print("=" * 40)
print()
print(classification_report(y_test, y_pred,
      target_names=['Tidak Diabetes', 'Diabetes']))

### 6.3 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tidak Diabetes', 'Diabetes'],
            yticklabels=['Tidak Diabetes', 'Diabetes'],
            annot_kws={'size': 16})
plt.title('Confusion Matrix — Random Forest Classifier', fontsize=13, fontweight='bold')
plt.ylabel('Label Aktual', fontsize=12)
plt.xlabel('Label Prediksi', fontsize=12)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negative  (TN): {tn}")
print(f"False Positive (FP): {fp}")
print(f"False Negative (FN): {fn}")
print(f"True Positive  (TP): {tp}")

### 6.4 Feature Importance

In [ ]:
importances = rf.feature_importances_
feature_names = list(X.columns)
sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
bars = plt.bar(range(len(feature_names)), importances[sorted_idx],
               color='steelblue', edgecolor='black', alpha=0.8)
plt.xticks(range(len(feature_names)),
           [feature_names[i] for i in sorted_idx], rotation=30, ha='right')
plt.title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
plt.ylabel('Importance Score')
plt.tight_layout()
plt.show()

print("Peringkat Feature Importance:")
for i in sorted_idx:
    print(f"  {feature_names[i]:28s}: {importances[i]:.4f}")

## 7. Simpan Model

In [ ]:
joblib.dump(rf, '../model/random_forest.pkl')
joblib.dump(kmeans, '../model/kmeans.pkl')
joblib.dump(scaler, '../model/scaler.pkl')
joblib.dump(risk_map, '../model/risk_map.pkl')

print("Model tersimpan di folder /model/:")
print("  ✓ random_forest.pkl")
print("  ✓ kmeans.pkl")
print("  ✓ scaler.pkl")
print("  ✓ risk_map.pkl")

## 8. Contoh Prediksi

In [ ]:
# Contoh prediksi pasien baru
sample = np.array([[6, 148, 72, 35, 0, 33.6, 0.627, 50]])

# Imputasi jika ada nilai 0
zero_cols_idx = [1, 2, 3, 4, 5]  # Glucose, BP, SkinThick, Insulin, BMI
for idx in zero_cols_idx:
    if sample[0, idx] == 0:
        sample[0, idx] = np.median(X[:, idx])

sample_scaled = scaler.transform(sample)
cluster_pred = kmeans.predict(sample_scaled)[0]
class_pred = rf.predict(sample_scaled)[0]
class_prob = rf.predict_proba(sample_scaled)[0][1]

print("=== CONTOH PREDIKSI PASIEN ===")
print(f"Input data   : {dict(zip(X.columns, sample[0]))}")
print()
print(f"Cluster      : {cluster_pred} → {risk_map[cluster_pred]}")
print(f"Prediksi     : {'Diabetes' if class_pred == 1 else 'Tidak Diabetes'}")
print(f"Probabilitas : {class_prob*100:.1f}% kemungkinan diabetes")